In [ ]:
import torch
import os

# Load both datasets from Kaggle input
pokemon_path = '/kaggle/input/datasets/jackemartin/pokemon-sprites/pokemon_images/pokemondb.net'
anime_path = '/kaggle/input/datasets/soumikrakshit/anime-faces/data'

pokemon_images = []
for root, dirs, files in os.walk(pokemon_path):
    pokemon_images.extend([os.path.join(root, f) for f in files if f.endswith(('.png', '.jpg', '.jpeg'))])

anime_images = []
for root, dirs, files in os.walk(anime_path):
    anime_images.extend([os.path.join(root, f) for f in files if f.endswith(('.png', '.jpg', '.jpeg'))])

print(f"✓ Loaded {len(pokemon_images)} Pokemon images")
print(f"✓ Loaded {len(anime_images)} Anime images")

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

class ImageDataset(Dataset):
    def __init__(self, img_paths, size=64):
        self.img_paths = img_paths
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])
    
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx])
        if img.mode == 'P':
            img = img.convert('RGBA')
        img = img.convert('RGB')
        return self.transform(img)

# Merge both datasets - use ALL images for better training
merged_images = pokemon_images + anime_images
train_loader = DataLoader(ImageDataset(merged_images), batch_size=128, shuffle=True, num_workers=4)
print(f"✓ Merged dataset: {len(merged_images)} total images ({len(pokemon_images)} Pokemon + {len(anime_images)} Anime)")
print(f"✓ DataLoader: {len(train_loader)} batches (batch_size=128)")
print(f"✓ Increased batch size for faster training")

In [ ]:
import torch.nn as nn

class DCGANGenerator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0), nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()
        )
    def forward(self, z):
        return self.main(z.view(-1, z.size(1), 1, 1))

class DCGANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, 1, 0)
        )
    def forward(self, x):
        return self.main(x).view(-1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dcgan_gen = DCGANGenerator().to(device)
dcgan_disc = DCGANDiscriminator().to(device)
print(f"✓ DCGAN models created (device: {device})")

In [ ]:
import torch.nn as nn

class WGANGPGenerator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0), nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()
        )
    def forward(self, z):
        return self.main(z.view(-1, z.size(1), 1, 1))

class WGANGPCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.InstanceNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.InstanceNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.InstanceNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 2, 1), nn.InstanceNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, 1, 0)
        )
    def forward(self, x):
        return self.main(x).view(-1)

wgan_gen = WGANGPGenerator().to(device)
wgan_critic = WGANGPCritic().to(device)
print("✓ WGAN-GP models created")